In [2]:
%load_ext autoreload
%autoreload 2

In [4]:
import sys
import pandas as pd
import altair as alt
sys.path.append("..")

In [5]:
# chargement des données
df = pd.read_csv("../industrialisation_perf/perf_avant_indus.csv")
df["requete"] = range(1, len(df) + 1)
df["gain"] = df["duration_before_indus"] - df["duration_after_indus"]
df

,duration_before_indus,duration_after_indus,requete,gain
0,4.57,3.24,1,1.33
1,1.90,1.56,2,0.34
2,3.60,1.03,3,2.57
3,4.71,1.37,4,3.34
4,4.02,0.90,5,3.12


In [6]:
# graphique : avant / après + gain
df_long = df.melt(
    id_vars="requete",
    value_vars=["duration_before_indus", "duration_after_indus"],
    var_name="statut",
    value_name="duration"
)

barres = alt.Chart(df_long).mark_bar(opacity=0.8).encode(
    x=alt.X("requete:O", title="Requête n°"),
    y=alt.Y("duration:Q", title="Durée (s)"),
    color=alt.Color(
        "statut:N",
        title="",
        scale=alt.Scale(
            domain=["duration_before_indus", "duration_after_indus"],
            range=["#d9534f", "#5cb85c"]
        ),
        legend=alt.Legend(labelExpr=(
            "datum.label === 'duration_before_indus' ? 'Avant industrialisation' : 'Après industrialisation'"
        ))
    ),
    xOffset="statut:N",
    tooltip=["requete:O", "statut:N", alt.Tooltip("duration:Q", format=".2f")]
)

texte_gain = alt.Chart(df).mark_text(dy=-8, fontSize=11, fontWeight="bold", color="#333").encode(
    x=alt.X("requete:O"),
    y=alt.Y("duration_before_indus:Q"),
    text=alt.Text("gain:Q", format=".2f")
)

(barres + texte_gain).properties(
    title="Gain de temps par requête (avant vs après industrialisation)",
    width=350
)

alt.LayerChart(...)

In [10]:
# graphique scatter : avant vs après avec zones colorées et droite x=y
import numpy as np

max_val = max(df["duration_before_indus"].max(), df["duration_after_indus"].max()) * 1.1
domaine = [0, max_val]

x_vals = np.linspace(0, max_val, 200)
zones_rows = []
for x in x_vals:
    zones_rows += [
        {"x": x, "y_top": x,       "y_bot": x * 3/4, "zone": "< 25%"},
        {"x": x, "y_top": x * 3/4, "y_bot": x / 2,   "zone": "25–50%"},
        {"x": x, "y_top": x / 2,   "y_bot": x / 4,   "zone": "50–75%"},
        {"x": x, "y_top": x / 4,   "y_bot": 0,        "zone": "> 75%"},
    ]

df_zones = pd.DataFrame(zones_rows)

zones = alt.Chart(df_zones).mark_area(opacity=0.18).encode(
    x=alt.X("x:Q", scale=alt.Scale(domain=domaine)),
    y=alt.Y("y_bot:Q", scale=alt.Scale(domain=domaine)),
    y2=alt.Y2("y_top:Q"),
    color=alt.Color(
        "zone:N",
        scale=alt.Scale(
            domain=["< 25%", "25–50%", "50–75%", "> 75%"],
            range=["#d9534f", "#f0ad4e", "#5bc0de", "#5cb85c"]
        ),
        legend=alt.Legend(title="Gain")
    )
)

# lignes de délimitation
def make_ligne(y2, color, dash):
    return alt.Chart(
        alt.InlineData(values=[{"x": 0, "y": 0}, {"x": max_val, "y": y2}])
    ).mark_line(strokeDash=dash, color=color, opacity=0.7, strokeWidth=1.2).encode(
        x=alt.X("x:Q", scale=alt.Scale(domain=domaine)),
        y=alt.Y("y:Q", scale=alt.Scale(domain=domaine))
    )

ligne_xy  = make_ligne(max_val,         "gray",    [6, 4])
ligne_25  = make_ligne(max_val * 3/4,   "#c87f0a", [4, 3])
ligne_50  = make_ligne(max_val / 2,     "#2a7b9b", [4, 3])
ligne_75  = make_ligne(max_val / 4,     "#3a7a3a", [4, 3])

# labels des lignes
def make_lbl(y_pos, texte, color):
    return alt.Chart(
        alt.InlineData(values=[{"x": max_val * 0.55, "y": y_pos, "label": texte}])
    ).mark_text(align="left", dx=4, dy=-6, fontSize=9, color=color).encode(
        x=alt.X("x:Q", scale=alt.Scale(domain=domaine)),
        y=alt.Y("y:Q", scale=alt.Scale(domain=domaine)),
        text="label:N"
    )

lbl_xy = make_lbl(max_val * 0.55,        "0% de gain",  "gray")
lbl_25 = make_lbl(max_val * 0.55 * 3/4,  "–25%",        "#c87f0a")
lbl_50 = make_lbl(max_val * 0.55 / 2,    "–50%",        "#2a7b9b")
lbl_75 = make_lbl(max_val * 0.55 / 4,    "–75%",        "#3a7a3a")

# points et étiquettes
points = alt.Chart(df).mark_circle(size=100, opacity=0.95, stroke="white", strokeWidth=1).encode(
    x=alt.X("duration_before_indus:Q", title="Avant industrialisation (s)", scale=alt.Scale(domain=domaine)),
    y=alt.Y("duration_after_indus:Q",  title="Après industrialisation (s)",  scale=alt.Scale(domain=domaine)),
    color=alt.value("#333"),
    tooltip=[
        alt.Tooltip("requete:O",               title="Requête n°"),
        alt.Tooltip("duration_before_indus:Q", title="Avant (s)",  format=".2f"),
        alt.Tooltip("duration_after_indus:Q",  title="Après (s)",  format=".2f"),
        alt.Tooltip("gain:Q",                  title="Gain (s)",   format=".2f"),
    ]
)

etiquettes = alt.Chart(df).mark_text(dx=10, fontSize=10, align="left").encode(
    x=alt.X("duration_before_indus:Q", scale=alt.Scale(domain=domaine)),
    y=alt.Y("duration_after_indus:Q",  scale=alt.Scale(domain=domaine)),
    text=alt.Text("requete:O")
)

(zones + ligne_xy + ligne_25 + ligne_50 + ligne_75
 + lbl_xy + lbl_25 + lbl_50 + lbl_75
 + points + etiquettes
).properties(
    title="Effet de l'industrialisation sur les performances de génération des diags",
    width=400,
    height=400
)

alt.LayerChart(...)